# Anomaly Detection Model Training

This notebook trains a scikit-learn model for vibration anomaly detection.
The model learns normal vibration patterns and flags anomalies when values exceed thresholds.

**Pipeline:**
1. Generate synthetic vibration training data
2. Train an Isolation Forest model
3. Save the model locally
4. Upload to MinIO S3 for serving via ModelMesh

In [ ]:
!pip install -q numpy scikit-learn joblib minio

In [ ]:
import numpy as np
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os

## 1. Generate Synthetic Vibration Data

Normal vibration range: 10-18 units (matching sensor config).
Anomalies: spikes above 30 units (matching `SENSOR_VIBRATION_PEAKINTETVAL`).

In [ ]:
np.random.seed(42)

n_normal = 5000
n_anomaly = 250

normal_data = np.random.uniform(10, 18, size=(n_normal, 1))
anomaly_data = np.random.uniform(30, 50, size=(n_anomaly, 1))

X = np.vstack([normal_data, anomaly_data])
y_true = np.array([1] * n_normal + [-1] * n_anomaly)  # 1=normal, -1=anomaly

shuffle_idx = np.random.permutation(len(X))
X = X[shuffle_idx]
y_true = y_true[shuffle_idx]

print(f"Dataset: {len(X)} samples ({n_normal} normal, {n_anomaly} anomalies)")
print(f"Normal range: [{normal_data.min():.1f}, {normal_data.max():.1f}]")
print(f"Anomaly range: [{anomaly_data.min():.1f}, {anomaly_data.max():.1f}]")

## 2. Train Isolation Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_true, test_size=0.2, random_state=42)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)
model.fit(X_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Anomaly', 'Normal']))

In [ ]:
test_values = np.array([[12.0], [15.0], [17.5], [35.0], [45.0]])
predictions = model.predict(test_values)
for val, pred in zip(test_values.flatten(), predictions):
    status = 'NORMAL' if pred == 1 else 'ANOMALY'
    print(f"Vibration {val:5.1f} → {status}")

## 3. Save Model

In [ ]:
model_dir = './models/sklearn/model/1'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'model.joblib')
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")
print(f"Size: {os.path.getsize(model_path) / 1024:.1f} KB")

## 4. Upload Model to MinIO S3

Uploads the model to the `anomaly-detection` bucket so ModelMesh can serve it.

In [ ]:
from minio import Minio
import urllib3

http_client = urllib3.PoolManager(cert_reqs='CERT_NONE')

s3_endpoint = os.environ.get('AWS_S3_ENDPOINT', 'minio.industrial-edge-ml-workspace.svc:9000')
s3_endpoint = s3_endpoint.replace('http://', '').replace('https://', '')
access_key = os.environ.get('AWS_ACCESS_KEY_ID', 'admin')
secret_key = os.environ.get('AWS_SECRET_ACCESS_KEY', 'admin123')
bucket = os.environ.get('AWS_S3_BUCKET', 'anomaly-detection')

client = Minio(
    s3_endpoint,
    access_key=access_key,
    secret_key=secret_key,
    secure=False,
    http_client=http_client
)

if not client.bucket_exists(bucket):
    client.make_bucket(bucket)
    print(f"Created bucket: {bucket}")

s3_path = 'sklearn/model/model.joblib'
client.fput_object(bucket, s3_path, model_path)
print(f"Uploaded {model_path} → s3://{bucket}/{s3_path}")

In [ ]:
print("\nFiles in bucket:")
for obj in client.list_objects(bucket, recursive=True):
    print(f"  {obj.object_name} ({obj.size} bytes)")

print("\nModel training and upload complete!")
print("The InferenceService can now serve predictions from ModelMesh.")